In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import warnings

# Gereksiz uyarı mesajlarını gizliyoruz
warnings.filterwarnings('ignore')

# Tekrarlanabilirlik (Reproducibility) için random seed sabitleniyor
np.random.seed(42) 

def load_and_preprocess_data(filepath):
    """
    CSV dosyasını okur, ön temizlik yapar (-200'leri NaN yapar, gereksiz sütunları atar).
    Eksik veri (NaN) satırlarını silmez, iki farklı senaryo için ham zemin hazırlar.

    Parametreler:
        filepath (str): Okunacak veri setinin dosya yolu.
    
    Döndürür:
        pd.DataFrame: İçinde NaN değerler barındıran, ön temizliği yapılmış sayısal veri seti.
    """
    print("=== 1. VERİ YÜKLEME VE ÖN TEMİZLİK ===")
    
    # Noktalı virgül ve virgüllü ondalık formatıyla okuma
    dataset = pd.read_csv(filepath, sep=';', decimal=',')
    dataset = dataset.dropna(how='all', axis=1).dropna(how='all', axis=0)
    print(f"[*] Orijinal Veri Boyutu: {dataset.shape}")

    # Sensör hata kodu olan -200'leri NaN (boşluk) ile değiştirme
    dataset = dataset.replace(-200, np.nan)
    
    # %90'ı eksik olan 'NMHC(GT)' sütunu ve metinsel sütunlar çıkarılır
    preprocessed_data = dataset.drop(columns=['NMHC(GT)', 'Date', 'Time'])
    
    return preprocessed_data


def summarize_data(data, scenario_name):
    """
    Veri setinin genel yapısını ekrana yazdırır.
    """
    print(f"\n=== 2. VERİ SETİ ÖZETİ ({scenario_name}) ===")
    print("[*] Veri Setinin İlk 5 Satırı:")
    print(data.head())
    print("\n[*] Sayısal Değişkenlerin İstatistiksel Özeti:")
    print(data.describe().T) 
    print("\n")


def plot_exploratory_data_analysis(data):
    """
    Veri seti üzerinde EDA (Keşifçi Veri Analizi) grafiklerini çizer.
    """
    print("=== 3. GÖRSELLEŞTİRME (EDA) ===")
    print("[*] Grafikler oluşturuluyor. Lütfen açılan pencereyi inceleyin...\n")
    
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(16, 12))

    sns.histplot(data['T'], kde=True, bins=30, color='royalblue', ax=axes[0, 0])
    axes[0, 0].set_title('Sıcaklık (T) Dağılımı', fontsize=14)

    sns.scatterplot(x='T', y='RH', data=data, alpha=0.5, color='darkorange', ax=axes[0, 1])
    axes[0, 1].set_title('Sıcaklık (T) ve Bağıl Nem (RH) İlişkisi', fontsize=14)

    sns.boxplot(x=data['CO(GT)'], color='mediumseagreen', ax=axes[1, 0])
    axes[1, 0].set_title('Karbonmonoksit (CO) Dağılımı ve Aykırı Değerler', fontsize=14)

    korelasyon = data[['CO(GT)', 'C6H6(GT)', 'NOx(GT)', 'NO2(GT)', 'T', 'RH']].corr()
    sns.heatmap(korelasyon, annot=True, cmap='viridis', fmt=".2f", linewidths=0.5, ax=axes[1, 1])
    axes[1, 1].set_title('Sensör Verileri Arası Korelasyon Haritası', fontsize=14)

    plt.tight_layout()
    plt.show()


def analyze_normal_distribution(data_series):
    """ Normal dağılım parametrelerini ve 68-95-99.7 kuralını raporlar. """
    mu = data_series.mean()
    sigma = data_series.std()
    print(f"[*] Ortalama (mu): {mu:.4f}")
    print(f"[*] Standart Sapma (sigma): {sigma:.4f}")

    z_scores = (data_series - mu) / sigma

    oran_1 = len(z_scores[(z_scores >= -1) & (z_scores <= 1)]) / len(z_scores) * 100
    oran_2 = len(z_scores[(z_scores >= -2) & (z_scores <= 2)]) / len(z_scores) * 100
    oran_3 = len(z_scores[(z_scores >= -3) & (z_scores <= 3)]) / len(z_scores) * 100

    print("\n[*] 68-95-99.7 Kuralı Kontrolü:")
    print(f"   ±1 Std Sapma: %{oran_1:.2f} | ±2 Std Sapma: %{oran_2:.2f} | ±3 Std Sapma: %{oran_3:.2f}")

    return z_scores, mu, sigma


def check_normality(data_series, scenario_name):
    """ Verinin normalliğini Shapiro-Wilk testi ve Q-Q Plot ile değerlendirir. """
    print("\n[*] NORMALLİK TESTİ (Shapiro-Wilk)")
    test_statistic, p_value = stats.shapiro(data_series)
    print(f"   -> Test İstatistiği: {test_statistic:.4f}, P-Değeri: {p_value:.4e}")
    
    if p_value > 0.05:
        print("   -> Sonuç: P > 0.05 (Veri normal dağılıma uyuyor).")
    else:
        print("   -> Sonuç: P < 0.05 (Veri normal dağılıma uymuyor).")

    plt.figure(figsize=(6, 4))
    stats.probplot(data_series, dist="norm", plot=plt)
    plt.title(f"Q-Q Plot: {scenario_name}", fontsize=12)
    plt.show()


def detect_anomalies_and_ci(data_series, z_scores, mu, sigma):
    """ Güven Aralığı oluşturur ve |Z| > 3 koşulunu sağlayan anomalileri bulur. """
    print("\n[*] GÜVEN ARALIĞI VE ANOMALİ TESPİTİ")
    
    n_samples = len(data_series)
    standard_error = sigma / np.sqrt(n_samples)
    ci_lower, ci_upper = stats.norm.interval(0.95, loc=mu, scale=standard_error)
    print(f"   -> %95 Güven Aralığı: ({ci_lower:.4f},  {ci_upper:.4f})")

    anomalies = data_series[abs(z_scores) > 3]
    print(f"   -> Tespit Edilen Anomali (|Z| > 3) Sayısı: {len(anomalies)}")


def run_statistical_pipeline(data_series, scenario_name):
    """ 
    İstatistiksel analiz adımlarını tek bir çatı altında toplayarak 
    farklı senaryolar için kolayca çağrılmasını sağlar. 
    """
    print("\n" + "="*60)
    print(f"=== İSTATİSTİKSEL ANALİZ: {scenario_name} ===")
    print("="*60)
    
    z_scores, mu, sigma = analyze_normal_distribution(data_series)
    check_normality(data_series, scenario_name)
    detect_anomalies_and_ci(data_series, z_scores, mu, sigma)


def main():
    """Ana orkestrasyon fonksiyonu."""
    file_path = 'AirQualityUCI.csv'
    target_col = 'T'
    
    # 1. Ham Veriyi Ön İşleme
    preprocessed_data = load_and_preprocess_data(file_path)
    
    # ---------------------------------------------------------
    # SENARYO 1: EKSİK VERİLERİ SİLME (Dropna)
    # ---------------------------------------------------------
    data_silinmis = preprocessed_data.dropna()
    print(f"[*] Senaryo 1 (Satırları Silinmiş) Veri Boyutu: {data_silinmis.shape}")
    
    # ---------------------------------------------------------
    # SENARYO 2: EKSİK VERİLERİ ORTALAMA İLE DOLDURMA (Imputation)
    # ---------------------------------------------------------
    data_ortalama = preprocessed_data.copy()
    for col in data_ortalama.columns:
        col_mean = data_ortalama[col].mean()
        data_ortalama[col] = data_ortalama[col].fillna(col_mean)
    print(f"[*] Senaryo 2 (Ortalama ile Doldurulmuş) Veri Boyutu: {data_ortalama.shape}\n")
    
    # EDA ve Özet sadece gerçek veri (Senaryo 1) üzerinden yapılır
    summarize_data(data_silinmis, "Senaryo 1 - Temizlenmiş")
    plot_exploratory_data_analysis(data_silinmis)
    
    # Her İki Senaryo İçin İstatistiksel Analiz Hattını Çalıştırma
    run_statistical_pipeline(data_silinmis[target_col], "SENARYO 1 (EKSİKLER SİLİNDİ)")
    run_statistical_pipeline(data_ortalama[target_col], "SENARYO 2 (ORTALAMA İLE DOLDURULDU)")

if __name__ == "__main__":
    main()